In [61]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score, precision_score, recall_score, mean_absolute_percentage_error

In [9]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [10]:
file_path = "WineQT.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/wine-quality-dataset",
  file_path,
)

print("First 5 records:", df.head())

/tmp/ipykernel_679/2456761086.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'wine-quality-dataset' dataset.
First 5 records:    fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.5

In [102]:
X = df.drop('quality', axis=1)
median_quality = 5
y = np.where(df["quality"] > median_quality, 1, -1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [111]:
# Классификация
X_train, X_test, y_train_cls, y_test_cls = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [129]:
def linear_svc(X, y, lr=0.01):
    iteration = 1000
    n, d = X.shape
    a = np.zeros(X.shape[1])
    b = 0.0
    for iter in range(iteration):
        for i in range(n):
            margin = y[i] * (np.dot(X[i], a) + b)
            if margin < 1:
                a -= lr * (a - y[i] * X[i])
                b -= lr * (-y[i])
            else:
                a -= lr * a
    return a, b

def predict_svc(X, a, b):
    return np.sign(np.dot(X, a) + b).astype(int)

In [120]:
a_svc, b_svc = linear_svc(X_train, y_train_cls)
pred_cls = predict_svc(X_test, a_svc, b_svc)

In [121]:
print("accuracy: ", accuracy_score(y_test_cls, pred_cls))
print("precision:", precision_score(y_test_cls, pred_cls))
print("recall:   ", recall_score(y_test_cls, pred_cls))
print("f1:       ", f1_score(y_test_cls, pred_cls))

accuracy:  0.777292576419214
precision: 0.8016528925619835
recall:    0.782258064516129
f1:        0.7918367346938775


In [122]:
# Регрессия
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [126]:
def linear_svr(X, y, lr=0.001):
    iteration = 1000
    n, d = X.shape
    a = np.zeros(X.shape[1])
    b = 0.0
    for iter in range(iteration):
        for i in range(n):
            pred = np.dot(X[i], a) + b
            error = y[i] - pred
            a += lr * error * X[i]
            b += lr * error
    return a, b

def predict_svr(X, a, b):
    return np.dot(X, a) + b

In [127]:
a_svr, b_svr = linear_svr(X_train_reg, y_train_reg)
pred_reg = predict_svr(X_test_reg, a_svr, b_svr)

r2 = r2_score(y_test_reg, pred_reg)
mean_error = np.abs(pred_reg - y_test_reg).mean()
mape = mean_absolute_percentage_error(y_test_reg, pred_reg)

In [128]:
print("R2 на тесте:", round(r2, 3))
print("средняя абсолютная ошибка:", round(mean_error, 3))
print("MAPE:", round(mape, 3))

R2 на тесте: 0.273
средняя абсолютная ошибка: 0.712
MAPE: 0.712
